# C13 — Publication des modèles de production sur Hugging Face (I7, S1)

Pour chacun des trois modèles produits par `c13_production_models.ipynb` (sortie attachée),
on crée un dépôt **public** `<HF_USERNAME>/<identifiant>` et on y envoie :

- `model.safetensors` : les poids, sans pickle ;
- `config.json` : classes, standardisation, température, encodeur et sa révision, empreinte
  des features ;
- `README.md` : la fiche du modèle (`docs/models/<identifiant>.md`), avec l'en-tête de
  Hugging Face (licence non commerciale héritée des données).

Le jeton vient du secret Kaggle `HF_TOKEN` (Add-ons → Secrets, à cocher pour ce notebook),
jamais écrit dans le notebook. La révision (hash du commit) de chaque dépôt est écrite dans
`hf_revisions.json`, à recopier dans `configs/registry.yaml`.

In [ ]:
import sys

BRANCH = "main"
# Nom d'utilisateur Hugging Face du propriétaire des dépôts (huggingface.co/<nom>).
HF_USERNAME = ""
REPO_URL = "https://github.com/mentalist011101/TP_SIGNAL_DS.git"
CLONE_DIR = "/kaggle/working/TP_SIGNAL_DS"

if not HF_USERNAME:
    raise RuntimeError("Indiquer HF_USERNAME avant de lancer la publication.")
!rm -rf {CLONE_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {CLONE_DIR}
# Kaggle tourne sous Python 3.13, au-delà de requires-python (<3.13) du dépôt.
%pip install -q --ignore-requires-python -e {CLONE_DIR}
sys.path.insert(0, CLONE_DIR + "/src")

## Jeton et modèles à publier

In [ ]:
import json
import shutil
from pathlib import Path

from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

from echolens.config import REPO_ROOT
from echolens.core.models.production_config import CONFIG_FILE, WEIGHTS_FILE, load_production_config
from echolens.training.production import load_production_plan

token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=token)
print("Connecté en tant que :", api.whoami()["name"])

found = sorted(Path("/kaggle/input").glob("**/production/echolens-sound-event-crnn/config.json"))
if len(found) != 1:
    raise RuntimeError("Attacher la sortie de echolens-c13-production-models")
PRODUCTION_DIR = found[0].parent.parent
plan = load_production_plan()
print("Modèles :", [planned.id for planned in plan.models], "depuis", PRODUCTION_DIR)

## Publication : un dépôt public par modèle

L'en-tête YAML du README dit à Hugging Face la licence, la tâche et, pour les têtes, le modèle
de base (l'encodeur de Whisper base, gelé). Le texte est la fiche du dépôt, sans modification.

In [ ]:
STAGING_DIR = Path("/kaggle/working/hf")
revisions = {}
for planned in plan.models:
    source = PRODUCTION_DIR / planned.id
    config = load_production_config(source / CONFIG_FILE)
    folder = STAGING_DIR / planned.id
    folder.mkdir(parents=True, exist_ok=True)
    shutil.copy(source / WEIGHTS_FILE, folder / WEIGHTS_FILE)
    shutil.copy(source / CONFIG_FILE, folder / CONFIG_FILE)
    # Fiche du modèle : docs/models/<identifiant>.md, précédée de l'en-tête de Hugging Face.
    header = [
        "---",
        f"license: {planned.license}",
        "language: [fr]",
        "pipeline_tag: audio-classification",
    ]
    header.append("tags: [audio, audio-classification, echolens, safetensors]")
    if config.encoder is not None:
        header.append(f"base_model: {config.encoder.checkpoint}")
    header.append("---")
    card = (REPO_ROOT / "docs" / "models" / f"{planned.id}.md").read_text(encoding="utf-8")
    readme = "\n".join(header) + "\n\n" + card
    (folder / "README.md").write_text(readme, encoding="utf-8")
    # Dépôt public, puis envoi du dossier en un seul commit.
    repo_id = f"{HF_USERNAME}/{planned.id}"
    api.create_repo(repo_id, repo_type="model", private=False, exist_ok=True)
    commit = api.upload_folder(
        repo_id=repo_id, folder_path=str(folder), commit_message=f"Publie {planned.id} (C13)"
    )
    revisions[planned.id] = {"weights": f"hf://{repo_id}", "revision": commit.oid}
    print(f"{repo_id} : révision {commit.oid}")

Path("/kaggle/working/hf_revisions.json").write_text(
    json.dumps(revisions, indent=2), encoding="utf-8"
)
shutil.rmtree(CLONE_DIR, ignore_errors=True)
print(json.dumps(revisions, indent=2))